In [ ]:
# -*- coding: utf-8 -*-
import random
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, avg, count, regexp_replace, lower, to_timestamp, month, year, coalesce, lit, date_trunc
from pyspark.sql.types import DoubleType, StringType
from pyspark.storagelevel import StorageLevel
import sys

# Inicialización de Spark
try:
    spark = SparkSession.builder \n        .appName("SentimentAnalysisPipeline") \n        .getOrCreate()
    print("Sesión de Spark iniciada correctamente.")
except Exception as e:
    print(f"ERROR: Fallo al iniciar la sesión de Spark: {e}", file=sys.stderr)
    sys.exit(1)

# --- 1. DEFINICIÓN DE UDFs Y FUNCIONES ---

def analyze_sentiment(text, raw_sentiment):
    """
    UDF simulada para el análisis de sentimiento.
    En un entorno real, esta función cargaría un modelo de Machine Learning (ej. VADER, BERT).
    Devuelve una puntuación flotante entre -1.0 (negativo) y 1.0 (positivo).
    """
    if raw_sentiment == 'positive':
        return random.uniform(0.5, 1.0)
    elif raw_sentiment == 'negative':
        return random.uniform(-1.0, -0.5)
    else:
        return random.uniform(-0.4, 0.4)

# Registro de la UDF
sentiment_udf = udf(analyze_sentiment, DoubleType())

# --- 2. INGESTA Y LIMPIEZA INICIAL ---

try:
    # Cargar los datos desde el Data Lake simulado (JSON)
    # Asume que el script data_simulator.py ya creó data/raw_tweets.json
    df = spark.read.json("data/raw_tweets.json")
    print(f"Datos cargados. Total de registros: {df.count()}")
except Exception as e:
    print(f"Error al cargar datos: {e}. Asegúrese de ejecutar data_simulator.py primero.", file=sys.stderr)
    sys.exit(1)

# Limpieza y Transformación (ETL)
df_processed = df.withColumn("tweet_date", to_timestamp(col("tweet_date"), "yyyy-MM-dd HH:mm:ss")) \n                 .withColumn("text_clean", lower(col("text"))) \n                 .withColumn("text_clean", regexp_replace(col("text_clean"), r'http\S+|@\S+|#\S+', '')) \n                 .withColumn("user_location", coalesce(col("user_location"), lit("Desconocido"))) \n                 .withColumn("sentiment_score", sentiment_udf(col("text_clean"), col("raw_sentiment"))) \n                 .withColumn("analysis_month", month(col("tweet_date"))) \n                 .withColumn("analysis_year", year(col("tweet_date"))) \n                 .select("tweet_id", "airline", "text", "sentiment_score", "analysis_month", "analysis_year", "user_location", "negativereason")

# --- 3. OPTIMIZACIÓN (CACHING Y PARTICIONAMIENTO) ---

# Aplicar Caching después de la transformación costosa (limpieza y UDF)
df_processed.persist(StorageLevel.MEMORY_ONLY)
print("DataFrame persistido en memoria para optimización.")

# --- 4. ANÁLISIS Y AGREGACIÓN ---

# Agregación 1: Sentimiento Promedio por Aerolínea
df_sentiment_summary = df_processed.groupBy("airline", "analysis_year", "analysis_month") \n                                     .agg(avg("sentiment_score").alias("avg_sentiment"),
                                          count("tweet_id").alias("total_tweets")) \n                                     .orderBy(col("avg_sentiment").asc())

print("\n--- Resultados del Análisis 1: Sentimiento Promedio por Aerolínea ---")
df_sentiment_summary.show(truncate=False)

# Agregación 2: Identificación de las razones negativas para la aerolínea con peor rendimiento
worst_airline = df_sentiment_summary.limit(1).collect()[0]['airline']

df_negative_reasons = df_processed.filter((col("airline") == worst_airline) & col("negativereason").isNotNull()) \n                                    .groupBy("negativereason") \n                                    .agg(count("tweet_id").alias("count")) \n                                    .orderBy(col("count").desc()) \n                                    .limit(5)

print(f"\n--- Resultados del Análisis 2: 5 Razones Negativas Principales para {worst_airline} ---")
df_negative_reasons.show(truncate=False)

# --- 5. ALMACENAMIENTO (SIMULACIÓN DE ESCRITURA EN CASSANDRA) ---

# En un entorno real, el código se vería así (requiere el conector de Cassandra):
# df_sentiment_summary.write \n#     .format("org.apache.spark.sql.cassandra") \n#     .mode("overwrite") \n#     .option("keyspace", "analytics_ks") \n#     .option("table", "airline_sentiment_metrics") \n#     .partitionBy("airline", "analysis_year") \n#     .save()
print("\n--- Simulación de escritura en Cassandra (Serving Layer) finalizada. ---")

# Detener Spark
df_processed.unpersist()
spark.stop()
